# Continuation: ResNet50 building-damage classification

You already extracted building crops. This notebook continues **AI Disaster Damage Mapping and Severity Estimation** on the Google Drive folder `resnet_dataset`:

https://drive.google.com/drive/folders/1Yeb7vfp4bOTb2i1gcT0HBOBPpEE0uMHH

1. Download chips with `python ml/download_xbd_drive.py`  
2. Inspect class imbalance on the Joint Damage Scale  
3. Stratified train / val / test splits  
4. Fine-tune ImageNet ResNet50 with ordinal-aware loss  
5. Report weighted F1 (the xView2 primary metric)  
6. Roll predictions into a tile-level severity index  

Run from the repo with `ml/` on `PYTHONPATH`, or copy these cells into Colab and mount the crop folder you saved.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.append(str(ROOT / "ml"))

import pandas as pd
from labels import CLASSES

manifest = ROOT / "data" / "xbd" / "manifest.csv"
df = pd.read_csv(manifest)
print(df.head())
print(df["label"].value_counts().reindex(CLASSES))

## Why weighted F1, not accuracy

This Drive subset is **damage-heavy**: `no-damage` is the smallest class. Full xBD is the opposite. Still use weighted F1, plus per-class F1 for `minor-damage` / `major-damage`. Those two middle classes are visually close — expect most confusion there.

In [ ]:
# Prepare splits then train. On CPU, freeze the backbone and keep epochs small.
# On Colab GPU, add --unfreeze-backbone and raise epochs.

%cd {ROOT / "ml"}
!python prepare_splits.py --manifest {manifest} --out {ROOT / "data" / "splits"}
# !python train.py --root {ROOT} --epochs 8 --freeze-backbone --batch-size 16
# !python evaluate.py --root {ROOT} --checkpoint {ROOT / "models" / "resnet50_best.pt"}

## Severity mapping

Each building is ordinal: no-damage = 0, minor = 1/3, major = 2/3, destroyed = 1. A tile severity index is 100 × the mean of those weights. Use it to rank neighborhoods for inspection, not as a dollar-loss estimate.

In [ ]:
from metrics import tile_severity_index, class_histogram

for disaster, part in df.groupby("disaster"):
    labels = part["label"].tolist()
    print(f"{disaster:28s}  n={len(labels):3d}  severity={tile_severity_index(labels):5.1f}  {class_histogram(labels)}")